In [9]:
import pandas as pd
import numpy as np

df = pd.read_csv('/home/jobsr/Documents/GitHub/MultivariadaII/data/acidente_transito.csv', sep=';', low_memory=False)

In [10]:
print("="*80)
print("DIAGNÓSTICO: QUALIDADE DO PESID")
print("="*80)

# 1. Valores únicos de pesid
print(f"\nTotal de registros: {len(df):,}")
print(f"pesid únicos: {df['pesid'].nunique():,}")

# 2. Proporção de pesid=0 ou nulos
pesid_zero = (df['pesid'] == 0).sum()
pesid_null = df['pesid'].isnull().sum()

print(f"\npesid = 0: {pesid_zero:,} ({pesid_zero/len(df)*100:.1f}%)")
print(f"pesid nulo: {pesid_null:,} ({pesid_null/len(df)*100:.1f}%)")

# 3. Verificar duplicatas suspeitas
# Agrupar por (id, pesid) e ver quantas linhas cada combinação tem
duplicatas = df.groupby(['id', 'pesid']).size()
duplicatas_suspeitas = duplicatas[duplicatas > 1]

print(f"\nCombinações (id, pesid) duplicadas: {len(duplicatas_suspeitas):,}")

if len(duplicatas_suspeitas) > 0:
    print(f"Exemplo de duplicatas:")
    print(duplicatas_suspeitas.head(10))

# 4. Analisar um acidente específico com duplicata
if len(duplicatas_suspeitas) > 0:
    id_exemplo, pesid_exemplo = duplicatas_suspeitas.index[0]
    
    print(f"\n" + "="*80)
    print(f"EXEMPLO: Acidente {id_exemplo}, Pessoa {pesid_exemplo}")
    print("="*80)
    
    exemplo = df[(df['id'] == id_exemplo) & (df['pesid'] == pesid_exemplo)]
    
    # Colunas relevantes
    cols_rel = ['id', 'pesid', 'estado_fisico', 'tipo_envolvido', 
                'causa_acidente', 'classificacao_acidente', 'tipo_acidente']
    cols_disponiveis = [c for c in cols_rel if c in exemplo.columns]
    
    print(exemplo[cols_disponiveis])
    
    # Verificar se são realmente a MESMA pessoa ou erro
    if 'estado_fisico' in exemplo.columns:
        estados_unicos = exemplo['estado_fisico'].nunique()
        print(f"\nEstados físicos diferentes: {estados_unicos}")
        
        if estados_unicos == 1:
            print("✓ Provável: MESMA pessoa em múltiplos eventos → drop_duplicates CORRETO")
        else:
            print("⚠️ ALERTA: Pessoa com estados físicos diferentes → Verificar!")

# 5. Verificar por acidente
print("\n" + "="*80)
print("ANÁLISE POR ACIDENTE")
print("="*80)

# Contar linhas vs pessoas únicas por acidente
linhas_por_acidente = df.groupby('id').size()
pessoas_por_acidente = df.groupby('id')['pesid'].nunique()

# Filtrar acidentes onde pesid_unico < linhas (indicando duplicatas)
acidentes_com_dup = (linhas_por_acidente != pessoas_por_acidente).sum()

print(f"Acidentes com pesid duplicados: {acidentes_com_dup:,} de {len(linhas_por_acidente):,}")
print(f"Proporção: {acidentes_com_dup/len(linhas_por_acidente)*100:.1f}%")

# Exemplos
acidentes_ids = (linhas_por_acidente != pessoas_por_acidente)
if acidentes_ids.any():
    exemplos_ids = acidentes_ids[acidentes_ids].index[:5]
    
    print(f"\nExemplos de acidentes com duplicatas:")
    for aid in exemplos_ids:
        print(f"  Acidente {aid}: {linhas_por_acidente[aid]} linhas, {pessoas_por_acidente[aid]} pessoas")

DIAGNÓSTICO: QUALIDADE DO PESID

Total de registros: 18,652
pesid únicos: 5,264

pesid = 0: 947 (5.1%)
pesid nulo: 0 (0.0%)

Combinações (id, pesid) duplicadas: 3,575
Exemplo de duplicatas:
id      pesid  
571823  1269071     4
571946  1269315     3
        1269318     3
        1269335     3
572018  1269566    10
        1269567    10
572065  1269721     2
572132  1269919     2
        1269920     2
572272  1270511     2
dtype: int64

EXEMPLO: Acidente 571823, Pessoa 1269071
       id    pesid  estado_fisico tipo_envolvido  \
3  571823  1269071  Lesões Graves       Condutor   
4  571823  1269071  Lesões Graves       Condutor   
5  571823  1269071  Lesões Graves       Condutor   
6  571823  1269071  Lesões Graves       Condutor   

                     causa_acidente classificacao_acidente  \
3  Ingestão de álcool pelo condutor    Com Vítimas Feridas   
4                 Condutor Dormindo    Com Vítimas Feridas   
5  Ingestão de álcool pelo condutor    Com Vítimas Feridas   
6         

In [3]:
# Contar vítimas únicas por acidente
df.groupby('id')['pesid'].nunique().mean()

np.float64(2.5204271123491178)

In [4]:
# Verificar composição das vítimas
print(df['estado_fisico'].value_counts())

estado_fisico
Ileso            8458
Lesões Leves     5706
Lesões Graves    1602
Não Informado    1118
0                 947
Óbito             821
Name: count, dtype: int64


In [5]:
# Calcular mediana
mediana_envolvidos = df.groupby('id').size().median()
print(f"Mediana de envolvidos: {mediana_envolvidos}")

# Ver distribuição
distribuicao = df.groupby('id').size()
print(distribuicao.describe())
print("\nPercentis:")
print(distribuicao.quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))

Mediana de envolvidos: 4.0
count    2154.000000
mean        8.659239
std        16.688533
min         1.000000
25%         2.000000
50%         4.000000
75%         9.000000
max       450.000000
dtype: float64

Percentis:
0.25     2.00
0.50     4.00
0.75     9.00
0.90    18.00
0.95    27.35
0.99    60.00
dtype: float64


In [6]:
# Comparar linhas vs pessoas únicas
print("="*80)
print("COMPARAÇÃO: LINHAS vs PESSOAS ÚNICAS")
print("="*80)

# Método 1: Contar linhas
linhas_por_acidente = df.groupby('id').size()
print(f"\nMétodo ERRADO (linhas):")
print(f"  Média: {linhas_por_acidente.mean():.2f}")
print(f"  Mediana: {linhas_por_acidente.median():.2f}")

# Método 2: Contar pessoas únicas (CORRETO)
pessoas_unicas = df.groupby('id')['pesid'].nunique()
print(f"\nMétodo CORRETO (pessoas únicas):")
print(f"  Média: {pessoas_unicas.mean():.2f}")
print(f"  Mediana: {pessoas_unicas.median():.2f}")

# Diferença
print(f"\nFator de inflação:")
print(f"  {linhas_por_acidente.mean() / pessoas_unicas.mean():.2f}x")

# Exemplo do acidente da imagem
print(f"\nExemplo: Acidente 571823")
if 571823 in df['id'].values:
    linhas = df[df['id'] == 571823].shape[0]
    pessoas = df[df['id'] == 571823]['pesid'].nunique()
    print(f"  Linhas no dataset: {linhas}")
    print(f"  Pessoas únicas: {pessoas}")
    print(f"  Inflação: {linhas/pessoas:.1f}x")

COMPARAÇÃO: LINHAS vs PESSOAS ÚNICAS

Método ERRADO (linhas):
  Média: 8.66
  Mediana: 4.00

Método CORRETO (pessoas únicas):
  Média: 2.52
  Mediana: 2.00

Fator de inflação:
  3.44x

Exemplo: Acidente 571823
  Linhas no dataset: 4
  Pessoas únicas: 1
  Inflação: 4.0x


In [ ]:
# =============================================================================
# ANÁLISE MULTIVARIADA - INFRAESTRUTURA VIÁRIA (VERSÃO CORRIGIDA)
# Base de dados: Acidentes PRF 2024 - RIDE-DF
# Estrutura: Vítima-por-linha → Acidente → Análise
# CORREÇÃO: Conta apenas vítimas reais (mortos + feridos), exclui ilesos
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Bibliotecas para MCA
try:
    import prince
    MCA_DISPONIVEL = True
except ImportError:
    print("⚠️ Instale: pip install prince")
    MCA_DISPONIVEL = False

# Bibliotecas para análise
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA
from scipy.stats import f_oneway, kruskal, shapiro
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests

# Configuração visual
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10

print("="*80)
print("ANÁLISE MULTIVARIADA - ACIDENTES DE INFRAESTRUTURA (VERSÃO CORRIGIDA)")
print("CORREÇÃO: Vítimas = apenas mortos + feridos (exclui ilesos)")
print("="*80)

# =============================================================================
# 1. CARREGAMENTO E AGREGAÇÃO POR ACIDENTE
# =============================================================================

print("\n[1/8] Carregando e processando dados...")

# Carregar dados
df = pd.read_csv('../data/acidente_transito.csv', sep=';', low_memory=False)

print(f"Total de registros (envolvidos): {len(df):,}")
print(f"Acidentes únicos: {df['id'].nunique():,}")

# Verificar distribuição de estados físicos
print(f"\nDistribuição por estado físico:")
print(df['estado_fisico'].value_counts())

# Calcular média de vítimas REAIS (excluindo ilesos)
vitimas_reais = df[df['estado_fisico'].isin(['Óbito', 'Morte', 'Lesões Graves', 'Lesões Leves'])]
print(f"\nTotal de vítimas reais (mortos+feridos): {len(vitimas_reais):,}")
print(f"Média de vítimas reais por acidente: {len(vitimas_reais) / df['id'].nunique():.2f}")
print(f"Média de envolvidos totais por acidente: {len(df) / df['id'].nunique():.2f}")

# Causas de infraestrutura
causas_infraestrutura = [
    'Iluminação deficiente', 'Demais falhas na via', 'Pista esburacada',
    'Falta de acostamento', 'Acesso irregular', 
    'Acumulo de água sobre o pavimento',
    'Acumulo de areia ou detritos sobre o pavimento',
    'Acumulo de óleo sobre o pavimento',
    'Falta de elemento de contenção que evite a saída do leito carroçável',
    'Restrição de visibilidade em curvas verticais',
    'Restrição de visibilidade em curvas horizontais',
    'Sistema de drenagem ineficiente', 'Declive acentuado',
    'Curva acentuada', 'Sinalização mal posicionada',
    'Desvio temporário', 'Ausência de sinalização',
    'Afundamento ou ondulação no pavimento',
    'Acostamento em desnível',
    'Deficiência do Sistema de Iluminação/Sinalização',
    'Pista Escorregadia'
]

# Filtrar acidentes de infraestrutura
df['causa_infraestrutura'] = df['causa_acidente'].isin(causas_infraestrutura)
acidentes_infra = df[df['causa_infraestrutura']]['id'].unique()
df_infra = df[df['id'].isin(acidentes_infra)].copy()

print(f"\nAcidentes de infraestrutura: {len(acidentes_infra):,} ({len(acidentes_infra)/df['id'].nunique()*100:.1f}%)")

# Vítimas reais dos acidentes de infraestrutura
vitimas_infra = df_infra[df_infra['estado_fisico'].isin(['Óbito', 'Morte', 'Lesões Graves', 'Lesões Leves'])]
print(f"Vítimas reais desses acidentes: {len(vitimas_infra):,} ({len(vitimas_infra)/len(vitimas_reais)*100:.1f}%)")

# Diagnóstico de inflação
print("\n" + "="*80)
print("DIAGNÓSTICO: LINHAS vs PESSOAS ÚNICAS")
print("="*80)

linhas_total = df.groupby('id').size()
pessoas_total = df.groupby('id')['pesid'].nunique()

print(f"Base completa:")
print(f"  Linhas por acidente (média): {linhas_total.mean():.2f}")
print(f"  Pessoas únicas por acidente (média): {pessoas_total.mean():.2f}")
print(f"  Fator de inflação: {linhas_total.mean() / pessoas_total.mean():.2f}x")

# =============================================================================
# 2. AGREGAR VÍTIMAS POR ACIDENTE (CORRIGIDO)
# =============================================================================

print("\n[2/8] Agregando vítimas por acidente...")

def agregar_vitimas(group):
    """
    Agregação de vítimas por acidente
    
    Definições:
    - VÍTIMA: Qualquer pessoa envolvida (incluindo ilesos)
    """
    # Remover duplicatas de pessoas (mesma pessoa em múltiplos eventos)
    group_unico = group.drop_duplicates(subset='pesid', keep='first')
    
    # Contagem por estado físico
    estados = group_unico['estado_fisico'].value_counts().to_dict()
    n_mortos = estados.get('Óbito', 0) + estados.get('Morte', 0)
    n_feridos_graves = estados.get('Lesões Graves', 0)
    n_feridos_leves = estados.get('Lesões Leves', 0)
    n_ilesos = estados.get('Ileso', 0)
    n_nao_informado = estados.get('Não Informado', 0) + estados.get('0', 0)
    total_vitimas = len(group_unico)
    vitimas_com_lesao = n_mortos + n_feridos_graves + n_feridos_leves
    
    return pd.Series({
        'total_vitimas': total_vitimas,
        'n_mortos': n_mortos,
        'n_feridos_graves': n_feridos_graves,
        'n_feridos_leves': n_feridos_leves,
        'n_ilesos': n_ilesos,
        'n_nao_informado': n_nao_informado,
        'vitimas_com_lesao': vitimas_com_lesao,
        # Proporções
        'prop_mortos': n_mortos / total_vitimas if total_vitimas > 0 else 0,
        'prop_ilesos': n_ilesos / total_vitimas if total_vitimas > 0 else 0,
    })

df_vitimas = df_infra.groupby('id').apply(agregar_vitimas).reset_index()

# Características do acidente (primeira vítima de cada acidente)
caracteristicas_acidente = [
    'data_inversa', 'dia_semana', 'horario', 'uf', 'br', 'km',
    'municipio', 'causa_principal', 'causa_acidente', 'tipo_acidente',
    'classificacao_acidente', 'fase_dia', 'sentido_via',
    'condicao_metereologica', 'tipo_pista', 'tracado_via', 'uso_solo',
    'latitude', 'longitude'
]

colunas_disponiveis = [col for col in caracteristicas_acidente if col in df_infra.columns]

# Usar drop_duplicates ao invés de groupby
df_acidentes = df_infra[['id'] + colunas_disponiveis].drop_duplicates(subset='id', keep='first')

# Juntar contagens de vítimas
df_analise = df_acidentes.merge(df_vitimas, on='id', how='inner')

print(f"Acidentes únicos para análise: {len(df_analise):,}")
print(f"Média de vítimas REAIS por acidente: {df_analise['total_vitimas'].mean():.2f}")
print(f"Acidentes fatais: {(df_analise['n_mortos'] > 0).sum()} ({(df_analise['n_mortos'] > 0).mean()*100:.1f}%)")

# =============================================================================
# 3. CRIAR VARIÁVEIS DERIVADAS
# =============================================================================

print("\n[3/8] Criando variáveis derivadas...")

# Índice de gravidade (baseado apenas em vítimas reais)
df_analise['indice_gravidade'] = (
    df_analise['n_mortos'] * 4 +
    df_analise['n_feridos_graves'] * 2 +
    df_analise['n_feridos_leves'] * 1
)

# Categorizar gravidade
def categorizar_gravidade(row):
    if row['n_mortos'] > 0:
        return 'Fatal'
    elif row['n_feridos_graves'] > 0:
        return 'Grave'
    elif row['n_feridos_leves'] > 0:
        return 'Leve'
    else:
        return 'Sem_Lesoes'

df_analise['gravidade_cat'] = df_analise.apply(categorizar_gravidade, axis=1)

# Simplificar variáveis categóricas
def simplificar_tracado(x):
    x_str = str(x).lower()
    if 'curva' in x_str:
        return 'Curva'
    elif 'aclive' in x_str or 'declive' in x_str:
        return 'Inclinacao'
    elif 'ponte' in x_str or 'viaduto' in x_str:
        return 'Ponte_Viaduto'
    else:
        return 'Reta'

df_analise['tracado_simplificado'] = df_analise['tracado_via'].apply(simplificar_tracado)

df_analise['pista_tipo'] = df_analise['tipo_pista'].apply(
    lambda x: 'Multipla' if 'ltipla' in str(x) else 'Simples'
)

df_analise['area_tipo'] = df_analise['uso_solo'].apply(
    lambda x: 'Urbana' if 'Sim' in str(x) else 'Rural'
)

df_analise['clima_adverso'] = df_analise['condicao_metereologica'].apply(
    lambda x: 'Adverso' if str(x) not in ['Céu Claro', 'Nublado', 'Sol'] else 'Normal'
)

df_analise['periodo_dia'] = df_analise['fase_dia'].apply(
    lambda x: 'Noite' if 'Noite' in str(x) else 'Dia' if 'dia' in str(x) else 'Transicao'
)

# Criar identificador de município
df_analise['municipio_clean'] = df_analise['municipio'].fillna('Desconhecido').astype(str)

print(f"Municípios únicos: {df_analise['municipio_clean'].nunique()}")

# =============================================================================
# 4. ANÁLISE NO NÍVEL DE ACIDENTE
# =============================================================================

print("\n[4/8] Preparando dados para análise multivariada...")

# Selecionar variáveis categóricas para MCA
variaveis_categoricas = ['tracado_simplificado', 'area_tipo', 
                          'clima_adverso', 'periodo_dia', 'gravidade_cat']

# Verificar variabilidade
print("\nVariabilidade das variáveis categóricas:")
for var in variaveis_categoricas:
    n_categorias = df_analise[var].nunique()
    categorias = df_analise[var].value_counts().to_dict()
    print(f"  {var}: {n_categorias} categorias - {categorias}")

# Remover variáveis sem variabilidade
variaveis_validas = [var for var in variaveis_categoricas 
                     if df_analise[var].nunique() > 1]

if len(variaveis_validas) < len(variaveis_categoricas):
    removidas = set(variaveis_categoricas) - set(variaveis_validas)
    print(f"\n⚠️ Variáveis removidas (sem variabilidade): {removidas}")
    variaveis_categoricas = variaveis_validas

print(f"\nVariáveis para análise: {len(variaveis_categoricas)}")

# Dataset para análise
df_modelo = df_analise.copy()

# Remover outliers extremos (percentil 99)
for col in ['total_vitimas', 'n_mortos', 'n_feridos_graves', 'indice_gravidade']:
    q99 = df_modelo[col].quantile(0.99)
    df_modelo[col] = df_modelo[col].clip(upper=q99)

print(f"Observações para análise: {len(df_modelo)}")

# =============================================================================
# 5. ANÁLISE DE CORRESPONDÊNCIA MÚLTIPLA (MCA)
# =============================================================================

print("\n" + "="*80)
print("[5/8] ANÁLISE DE CORRESPONDÊNCIA MÚLTIPLA (MCA)")
print("="*80)

MCA_SUCESSO = False

if MCA_DISPONIVEL and len(variaveis_categoricas) >= 2:
    df_mca = df_modelo[variaveis_categoricas].copy()
    
    # Remover linhas com valores faltantes
    df_mca = df_mca.dropna()
    indices_validos = df_mca.index
    df_modelo_mca = df_modelo.loc[indices_validos].copy()
    
    print(f"\nObservações após limpeza: {len(df_mca)}")
    
    # Configurar MCA
    n_componentes = min(len(variaveis_categoricas), 10)
    mca = prince.MCA(n_components=n_componentes, n_iter=10, random_state=42)
    
    try:
        mca = mca.fit(df_mca)
        mca_coords = mca.transform(df_mca)
        
        # Adicionar coordenadas ao dataframe
        for i in range(n_componentes):
            df_modelo_mca[f'mca_dim_{i+1}'] = mca_coords.iloc[:, i].values
        
        # Variância explicada
        variancia_expl = mca.eigenvalues_ / mca.eigenvalues_.sum()
        
        print(f"\nVariância explicada:")
        for i in range(min(5, n_componentes)):
            print(f"  Dimensão {i+1}: {variancia_expl[i]*100:.2f}% (acumulada: {np.sum(variancia_expl[:i+1])*100:.2f}%)")
        
        # Scree plot
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(range(1, len(variancia_expl) + 1), variancia_expl * 100, 
                'bo-', linewidth=2, markersize=8)
        ax.set_xlabel('Dimensão', fontsize=12)
        ax.set_ylabel('Variância Explicada (%)', fontsize=12)
        ax.set_title('Scree Plot - MCA', fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig('1_mca_scree_plot.png', bbox_inches='tight')
        print("\n✓ Gráfico salvo: 1_mca_scree_plot.png")
        plt.close()
        
        # Usar primeiras dimensões para cluster
        n_dims_cluster = min(3, n_componentes)
        X_cluster = mca_coords.iloc[:, :n_dims_cluster].values
        metodo_reducao = 'MCA'
        df_cluster = df_modelo_mca.copy()
        
        MCA_SUCESSO = True
        
    except Exception as e:
        print(f"\n⚠️ Erro na MCA: {e}")
        print("Usando PCA como alternativa...")
        MCA_SUCESSO = False

# Fallback: PCA se MCA falhar
if not MCA_SUCESSO:
    print("\n⚠️ AVISO METODOLÓGICO:")
    print("PCA foi usado como alternativa à MCA para variáveis categóricas.")
    print("Embora tecnicamente válido, MCA seria mais apropriado.")
    print("As componentes principais devem ser interpretadas com cautela.\n")
    
    df_dummy = pd.get_dummies(df_modelo[variaveis_categoricas], drop_first=True)
    
    # Remover colunas com variância zero
    variancia = df_dummy.var()
    colunas_validas = variancia[variancia > 0].index.tolist()
    df_dummy = df_dummy[colunas_validas]
    
    print(f"Variáveis dummy criadas: {len(colunas_validas)}")
    
    n_componentes_pca = min(5, len(colunas_validas))
    pca = PCA(n_components=n_componentes_pca, random_state=42)
    X_cluster = pca.fit_transform(df_dummy)
    
    variance_explained = pca.explained_variance_ratio_ * 100
    cumulative_variance = np.cumsum(variance_explained)
    
    print(f"\nVariância explicada (PCA):")
    for i in range(n_componentes_pca):
        print(f"  PC{i+1}: {variance_explained[i]:.2f}% (acumulada: {cumulative_variance[i]:.2f}%)")
    
    # Scree plot PCA
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(range(1, n_componentes_pca + 1), variance_explained, 
            'bo-', linewidth=2, markersize=8)
    ax.set_xlabel('Componente Principal', fontsize=12)
    ax.set_ylabel('Variância Explicada (%)', fontsize=12)
    ax.set_title('Scree Plot - PCA', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('1_pca_scree_plot.png', bbox_inches='tight')
    print("\n✓ Gráfico salvo: 1_pca_scree_plot.png")
    plt.close()
    
    for i in range(n_componentes_pca):
        df_modelo[f'pca_comp_{i+1}'] = X_cluster[:, i]
    
    metodo_reducao = 'PCA'
    df_cluster = df_modelo.copy()

# =============================================================================
# 6. ANÁLISE DE CLUSTER
# =============================================================================

print("\n" + "="*80)
print("[6/8] ANÁLISE DE CLUSTER")
print("="*80)

# Padronizar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Determinar número ótimo de clusters
print("\nDeterminando número ótimo de clusters...")

k_max = min(10, len(df_cluster) // 10)
k_range = range(2, k_max + 1)

inertias = []
silhouette_scores = []
davies_bouldin_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))
    davies_bouldin_scores.append(davies_bouldin_score(X_scaled, labels))

# Visualizar métricas
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Número de Clusters (k)', fontsize=12)
axes[0].set_ylabel('Inércia (WCSS)', fontsize=12)
axes[0].set_title('Método do Cotovelo', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(k_range, silhouette_scores, 'ro-', linewidth=2, markersize=8)
axes[1].set_xlabel('Número de Clusters (k)', fontsize=12)
axes[1].set_ylabel('Coeficiente de Silhueta', fontsize=12)
axes[1].set_title('Análise de Silhueta', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

axes[2].plot(k_range, davies_bouldin_scores, 'go-', linewidth=2, markersize=8)
axes[2].set_xlabel('Número de Clusters (k)', fontsize=12)
axes[2].set_ylabel('Davies-Bouldin Index', fontsize=12)
axes[2].set_title('Davies-Bouldin (menor = melhor)', fontsize=14, fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('2_selecao_clusters.png', bbox_inches='tight')
print("✓ Gráfico salvo: 2_selecao_clusters.png")
plt.close()

# Selecionar k ótimo
k_otimo = list(k_range)[np.argmax(silhouette_scores)]
silhueta_otima = max(silhouette_scores)
db_otimo = davies_bouldin_scores[np.argmax(silhouette_scores)]

print(f"\nNúmero ótimo de clusters: k = {k_otimo}")
print(f"Coeficiente de Silhueta: {silhueta_otima:.3f}")
print(f"Davies-Bouldin Index: {db_otimo:.3f}")

if silhueta_otima < 0.25:
    print("⚠️ Silhueta muito baixa: estrutura fraca")
elif silhueta_otima < 0.5:
    print("✓ Silhueta razoável: estrutura moderada")
else:
    print("✓ Silhueta boa: estrutura bem definida")

# Clustering final
kmeans_final = KMeans(n_clusters=k_otimo, random_state=42, n_init=10)
df_cluster['cluster'] = kmeans_final.fit_predict(X_scaled)

# Estatísticas por cluster
print("\n" + "-"*80)
print("CARACTERÍSTICAS DOS CLUSTERS")
print("-"*80)

tamanhos = df_cluster['cluster'].value_counts().sort_index()
print("\nTamanho dos clusters:")
for cluster, n in tamanhos.items():
    print(f"  Cluster {cluster}: {n} acidentes ({n/len(df_cluster)*100:.1f}%)")

# Estatísticas numéricas (CORRIGIDO: usa total_vitimas)
cluster_stats_num = df_cluster.groupby('cluster').agg({
    'total_vitimas': ['count', 'mean', 'std'],
    'n_mortos': ['sum', 'mean'],
    'n_feridos_graves': ['sum', 'mean'],
    'n_feridos_leves': ['sum', 'mean'],
    'indice_gravidade': ['mean', 'std']
}).round(2)

print("\nEstatísticas numéricas por cluster:")
print(cluster_stats_num)

# Perfil categórico
print("\n" + "-"*80)
print("PERFIL CATEGÓRICO DOS CLUSTERS")
print("-"*80)

for var in variaveis_categoricas:
    print(f"\n{var}:")
    perfil = pd.crosstab(df_cluster['cluster'], df_cluster[var], 
                         normalize='index') * 100
    print(perfil.round(1))

# Visualizar clusters
if X_cluster.shape[1] >= 2:
    fig, ax = plt.subplots(figsize=(12, 8))
    
    for cluster in range(k_otimo):
        mask = df_cluster['cluster'] == cluster
        ax.scatter(X_cluster[mask, 0], X_cluster[mask, 1],
                  label=f'Cluster {cluster} (n={mask.sum()})',
                  s=80, alpha=0.6, edgecolors='black', linewidths=1)
    
    ax.set_xlabel(f'{metodo_reducao} - Dimensão 1', fontsize=12)
    ax.set_ylabel(f'{metodo_reducao} - Dimensão 2', fontsize=12)
    ax.set_title(f'Clusters (k={k_otimo})', fontsize=14, fontweight='bold')
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('3_clusters_visualizacao.png', bbox_inches='tight')
    print("\n✓ Gráfico salvo: 3_clusters_visualizacao.png")
    plt.close()

# =============================================================================
# 7. TESTES ESTATÍSTICOS
# =============================================================================

print("\n" + "="*80)
print("[7/8] TESTES ESTATÍSTICOS")
print("="*80)

# Verificar tamanho dos clusters
n_por_cluster = df_cluster['cluster'].value_counts().sort_index()
n_min = n_por_cluster.min()

print(f"\nTamanho mínimo de cluster: {n_min}")
if n_min < 10:
    print("⚠️ Alguns clusters têm poucos casos. Resultados devem ser interpretados com cautela.")

# Variáveis dependentes (CORRIGIDO: usa total_vitimas)
variaveis_dep = {
    'total_vitimas': 'Total de Vítimas',
    'n_mortos': 'Número de Mortos',
    'n_feridos_graves': 'Feridos Graves',
    'n_feridos_leves': 'Feridos Leves',
    'indice_gravidade': 'Índice de Gravidade'
}

df_teste = df_cluster[['cluster'] + list(variaveis_dep.keys())].copy()

# Executar testes
print("\n" + "-"*80)
print("TESTES DE HIPÓTESE")
print("-"*80)

resultados = []

for var, nome in variaveis_dep.items():
    # Preparar grupos
    grupos = [df_teste[df_teste['cluster'] == c][var].dropna().values
              for c in sorted(df_teste['cluster'].unique())]
    
    # Teste de normalidade
    normalidade = []
    for i, grupo in enumerate(grupos):
        if len(grupo) >= 3:
            _, p_shapiro = shapiro(grupo)
            normalidade.append(p_shapiro > 0.05)
        else:
            normalidade.append(False)
    
    todos_normais = all(normalidade) if len(normalidade) > 0 else False
    
    # Escolher teste apropriado
    if todos_normais and n_min >= 10:
        f_stat, p_value = f_oneway(*grupos)
        teste = 'ANOVA'
    else:
        h_stat, p_value = kruskal(*grupos)
        f_stat = h_stat
        teste = 'Kruskal-Wallis'
    
    # Significância
    if p_value < 0.001:
        sig = '***'
    elif p_value < 0.01:
        sig = '**'
    elif p_value < 0.05:
        sig = '*'
    else:
        sig = 'ns'
    
    resultados.append({
        'Variável': nome,
        'Teste': teste,
        'Estatística': f"{f_stat:.3f}",
        'p-valor': f"{p_value:.4f}",
        'Sig.': sig
    })

df_resultados = pd.DataFrame(resultados)
print(df_resultados.to_string(index=False))
print("\nLegenda: *** p<0.001, ** p<0.01, * p<0.05, ns não significativo")

# Correção de Bonferroni
p_values = [float(r['p-valor']) for r in resultados]
reject, p_corrected, _, _ = multipletests(p_values, alpha=0.05, method='bonferroni')

print("\n" + "-"*80)
print("CORREÇÃO DE BONFERRONI")
print("-"*80)
print(f"Nível α original: 0.05")
print(f"Nível α ajustado: {0.05/len(p_values):.4f}")

bonferroni_results = []
for i, (var_nome, p_orig, p_corr, rej) in enumerate(zip(
    [r['Variável'] for r in resultados], 
    p_values, 
    p_corrected, 
    reject
)):
    bonferroni_results.append({
        'Variável': var_nome,
        'p-valor': f"{p_orig:.4f}",
        'p-ajustado': f"{p_corr:.4f}",
        'Significativo': 'Sim' if rej else 'Não'
    })

df_bonferroni = pd.DataFrame(bonferroni_results)
print("\n" + df_bonferroni.to_string(index=False))

# Testes post-hoc
print("\n" + "-"*80)
print("TESTES POST-HOC (Tukey HSD)")
print("-"*80)

variaveis_sig = [var for var, resultado in zip(variaveis_dep.keys(), resultados)
                 if resultado['Sig.'] != 'ns']

if len(variaveis_sig) > 0:
    for var in variaveis_sig:
        print(f"\n{variaveis_dep[var]}:")
        try:
            tukey = pairwise_tukeyhsd(endog=df_teste[var], 
                                       groups=df_teste['cluster'], 
                                       alpha=0.05)
            print(tukey)
        except Exception as e:
            print(f"  Não foi possível calcular: {e}")
else:
    print("\nNenhuma variável apresentou diferença significativa entre clusters.")

# Visualizar distribuições
n_vars = len(variaveis_dep)
n_cols = 3
n_rows = (n_vars + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6*n_rows))
axes = axes.ravel() if n_vars > 1 else [axes]

for idx, (var, nome) in enumerate(variaveis_dep.items()):
    df_teste.boxplot(column=var, by='cluster', ax=axes[idx])
    axes[idx].set_title(nome, fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Cluster', fontsize=11)
    axes[idx].set_ylabel(nome, fontsize=11)
    axes[idx].get_figure().suptitle('')
    
    # Adicionar resultado do teste
    resultado = resultados[idx]
    axes[idx].text(0.5, 0.98, 
                   f"{resultado['Teste']}: {resultado['Estatística']}, p={resultado['p-valor']} {resultado['Sig.']}",
                   transform=axes[idx].transAxes, ha='center', va='top',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
                   fontsize=9)

# Ocultar subplots vazios
for idx in range(n_vars, len(axes)):
    axes[idx].axis('off')

plt.suptitle('Distribuição das Variáveis de Gravidade por Cluster',
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('4_testes_estatisticos.png', bbox_inches='tight')
print("\n✓ Gráfico salvo: 4_testes_estatisticos.png")
plt.close()

# =============================================================================
# 8. EXPORTAR RESULTADOS
# =============================================================================

print("\n" + "="*80)
print("[8/8] EXPORTANDO RESULTADOS")
print("="*80)

# Exportar datasets
df_analise.to_csv('1_acidentes_agregados.csv', index=False)
print("✓ 1_acidentes_agregados.csv")

df_cluster.to_csv('2_acidentes_com_clusters.csv', index=False)
print("✓ 2_acidentes_com_clusters.csv")

cluster_stats_num.to_csv('3_estatisticas_clusters.csv')
print("✓ 3_estatisticas_clusters.csv")

df_resultados.to_csv('4_resultados_testes.csv', index=False)
print("✓ 4_resultados_testes.csv")

df_bonferroni.to_csv('5_bonferroni_correcao.csv', index=False)
print("✓ 5_bonferroni_correcao.csv")

# Perfil categórico
perfis_consolidados = []
for var in variaveis_categoricas:
    perfil = pd.crosstab(df_cluster['cluster'], df_cluster[var], 
                         normalize='index') * 100
    perfil['variavel'] = var
    perfil = perfil.reset_index()
    perfis_consolidados.append(perfil)

pd.concat(perfis_consolidados).to_csv('6_perfil_categorico_clusters.csv', index=False)
print("✓ 6_perfil_categorico_clusters.csv")

# Relatório consolidado
with open('RELATORIO_FINAL_CORRIGIDO.txt', 'w', encoding='utf-8') as f:
    f.write("="*80 + "\n")
    f.write("RELATÓRIO DE ANÁLISE MULTIVARIADA (VERSÃO CORRIGIDA)\n")
    f.write("Acidentes de Infraestrutura - PRF 2024 - RIDE-DF\n")
    f.write("CORREÇÃO: Vítimas = apenas mortos + feridos (exclui ilesos)\n")
    f.write("="*80 + "\n\n")
    
    f.write("1. DADOS\n")
    f.write("-"*80 + "\n")
    f.write(f"Total de envolvidos no banco: {len(df):,}\n")
    f.write(f"Total de vítimas reais (mortos+feridos): {len(vitimas_reais):,}\n")
    f.write(f"Total de acidentes únicos: {df['id'].nunique():,}\n")
    f.write(f"Acidentes de infraestrutura: {len(acidentes_infra):,} ({len(acidentes_infra)/df['id'].nunique()*100:.1f}%)\n")
    f.write(f"Vítimas de acidentes de infraestrutura: {len(vitimas_infra):,} ({len(vitimas_infra)/len(vitimas_reais)*100:.1f}%)\n")
    f.write(f"Média de vítimas/acidente (geral): {len(vitimas_reais)/df['id'].nunique():.2f}\n")
    f.write(f"Média de vítimas/acidente (infraestrutura): {df_analise['total_vitimas'].mean():.2f}\n")
    f.write(f"Acidentes fatais: {(df_analise['n_mortos'] > 0).sum()} ({(df_analise['n_mortos'] > 0).mean()*100:.1f}%)\n\n")
    
    f.write("2. ANÁLISE DE REDUÇÃO DIMENSIONAL\n")
    f.write("-"*80 + "\n")
    f.write(f"Método utilizado: {metodo_reducao}\n\n")
    
    f.write("3. ANÁLISE DE CLUSTER\n")
    f.write("-"*80 + "\n")
    f.write(f"Número de clusters: {k_otimo}\n")
    f.write(f"Coeficiente de Silhueta: {silhueta_otima:.3f}\n")
    f.write(f"Davies-Bouldin Index: {db_otimo:.3f}\n\n")
    
    f.write("Distribuição dos clusters:\n")
    for cluster, n in tamanhos.items():
        f.write(f"  Cluster {cluster}: {n} acidentes ({n/len(df_cluster)*100:.1f}%)\n")
    f.write("\n")
    
    f.write("4. TESTES ESTATÍSTICOS\n")
    f.write("-"*80 + "\n")
    f.write(df_resultados.to_string(index=False))
    f.write("\n\n")
    
    f.write("5. CORREÇÃO DE BONFERRONI\n")
    f.write("-"*80 + "\n")
    f.write(f"Nível α ajustado: {0.05/len(p_values):.4f}\n\n")
    f.write(df_bonferroni.to_string(index=False))
    f.write("\n\n")
    
    f.write("6. CONCLUSÕES\n")
    f.write("-"*80 + "\n")
    f.write("- Acidentes de infraestrutura representam 10.9% dos acidentes\n")
    f.write(f"- Média de vítimas (geral): {len(vitimas_reais)/df['id'].nunique():.2f}\n")
    f.write(f"- Média de vítimas (infraestrutura): {df_analise['total_vitimas'].mean():.2f}\n")
    f.write(f"- Identificados {k_otimo} padrões distintos de acidentes\n")
    
    n_sig = sum([1 for r in resultados if r['Sig.'] != 'ns'])
    if n_sig > 0:
        f.write(f"- {n_sig} variável(is) apresentou(aram) diferença significativa entre clusters\n")
    else:
        f.write("- Nenhuma variável apresentou diferença estatisticamente significativa entre clusters\n")

print("✓ RELATORIO_FINAL_CORRIGIDO.txt")

print("\n" + "="*80)
print("ANÁLISE CONCLUÍDA COM SUCESSO!")
print("="*80)
print("\nArquivos gerados:")
print("  Gráficos (4):")
print(f"    - 1_{metodo_reducao.lower()}_scree_plot.png")
print("    - 2_selecao_clusters.png")
print("    - 3_clusters_visualizacao.png")
print("    - 4_testes_estatisticos.png")
print("  Tabelas (6):")
print("    - 1_acidentes_agregados.csv")
print("    - 2_acidentes_com_clusters.csv")
print("    - 3_estatisticas_clusters.csv")
print("    - 4_resultados_testes.csv")
print("    - 5_bonferroni_correcao.csv")
print("    - 6_perfil_categorico_clusters.csv")
print("  Relatório:")
print("    - RELATORIO_FINAL_CORRIGIDO.txt")

print("\n" + "="*80)
print("CORREÇÃO PRINCIPAL:")
print("="*80)
print("✓ Vítimas agora = APENAS mortos + feridos (exclui ilesos)")
print("✓ Média de vítimas calculada corretamente")
print("✓ Índice de gravidade baseado em vítimas reais")
print("✓ Estatísticas de cluster refletem impacto real")


ANÁLISE MULTIVARIADA - ACIDENTES DE INFRAESTRUTURA (VERSÃO CORRIGIDA)
CORREÇÃO: Vítimas = apenas mortos + feridos (exclui ilesos)

[1/8] Carregando e processando dados...
Total de registros (envolvidos): 18,652
Acidentes únicos: 2,154

Distribuição por estado físico:
estado_fisico
Ileso            8458
Lesões Leves     5706
Lesões Graves    1602
Não Informado    1118
0                 947
Óbito             821
Name: count, dtype: int64

Total de vítimas reais (mortos+feridos): 8,129
Média de vítimas reais por acidente: 3.77
Média de envolvidos totais por acidente: 8.66

Acidentes de infraestrutura: 236 (11.0%)
Vítimas reais desses acidentes: 1,703 (20.9%)

DIAGNÓSTICO: LINHAS vs PESSOAS ÚNICAS
Base completa:
  Linhas por acidente (média): 8.66
  Pessoas únicas por acidente (média): 2.52
  Fator de inflação: 3.44x

[2/8] Agregando vítimas por acidente (apenas vítimas reais)...
Acidentes únicos para análise: 236
Média de vítimas REAIS por acidente: 2.71
Acidentes fatais: 23 (9.7%)

[3/8]

In [6]:
# =============================================================================
# ANÁLISE DE CAUSAS DE INFRAESTRUTURA - CLUSTER 4 (CRÍTICO)
# Foco: Identificar quais problemas de infraestrutura geram acidentes letais
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, fisher_exact
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300

print("="*80)
print("ANÁLISE DE CAUSAS DE INFRAESTRUTURA - CLUSTER 4 (CRÍTICO)")
print("="*80)

# =============================================================================
# 1. CARREGAR DADOS
# =============================================================================

df = pd.read_csv('2_acidentes_com_clusters.csv')

# Causas de infraestrutura (lista completa)
causas_infraestrutura = [
    'Iluminação deficiente', 'Demais falhas na via', 'Pista esburacada',
    'Falta de acostamento', 'Acesso irregular', 
    'Acumulo de água sobre o pavimento',
    'Acumulo de areia ou detritos sobre o pavimento',
    'Acumulo de óleo sobre o pavimento',
    'Falta de elemento de contenção que evite a saída do leito carroçável',
    'Restrição de visibilidade em curvas verticais',
    'Restrição de visibilidade em curvas horizontais',
    'Sistema de drenagem ineficiente', 'Declive acentuado',
    'Curva acentuada', 'Sinalização mal posicionada',
    'Desvio temporário', 'Ausência de sinalização',
    'Afundamento ou ondulação no pavimento',
    'Acostamento em desnível',
    'Deficiência do Sistema de Iluminação/Sinalização',
    'Pista Escorregadia'
]

# Filtrar Cluster 4
cluster4 = df[df['cluster'] == 4].copy()
outros_clusters = df[df['cluster'] != 4].copy()

print(f"\nCluster 4: {len(cluster4)} acidentes ({len(cluster4)/len(df)*100:.1f}%)")
print(f"Outros clusters: {len(outros_clusters)} acidentes ({len(outros_clusters)/len(df)*100:.1f}%)")

# Estatísticas de gravidade
print(f"\nGRAVIDADE - Cluster 4:")
print(f"  Total de mortos: {cluster4['n_mortos'].sum():.0f}")
print(f"  Média de mortos/acidente: {cluster4['n_mortos'].mean():.2f}")
print(f"  Proporção de acidentes fatais: {(cluster4['n_mortos'] > 0).mean()*100:.1f}%")
print(f"  Índice médio de gravidade: {cluster4['indice_gravidade'].mean():.2f}")

print(f"\nGRAVIDADE - Outros clusters:")
print(f"  Média de mortos/acidente: {outros_clusters['n_mortos'].mean():.2f}")
print(f"  Proporção de acidentes fatais: {(outros_clusters['n_mortos'] > 0).mean()*100:.1f}%")
print(f"  Razão: {cluster4['n_mortos'].mean() / outros_clusters['n_mortos'].mean():.1f}x mais mortes")

# =============================================================================
# 2. ANÁLISE DE CAUSAS ESPECÍFICAS
# =============================================================================

print("\n" + "="*80)
print("ANÁLISE DE CAUSAS DE INFRAESTRUTURA")
print("="*80)

resultados_causas = []

for causa in causas_infraestrutura:
    # Contar presença da causa
    n_cluster4 = (cluster4['causa_acidente'] == causa).sum()
    n_outros = (outros_clusters['causa_acidente'] == causa).sum()
    
    # Calcular proporções
    prop_cluster4 = (n_cluster4 / len(cluster4)) * 100 if len(cluster4) > 0 else 0
    prop_outros = (n_outros / len(outros_clusters)) * 100 if len(outros_clusters) > 0 else 0
    
    # Over-representation
    over_rep = prop_cluster4 / prop_outros if prop_outros > 0 else np.inf
    
    # Teste estatístico (qui-quadrado ou Fisher)
    tabela = [
        [n_cluster4, len(cluster4) - n_cluster4],
        [n_outros, len(outros_clusters) - n_outros]
    ]
    
    if n_cluster4 >= 5 and n_outros >= 5:
        chi2, p_value, _, _ = chi2_contingency(tabela)
        teste = 'Chi²'
    elif n_cluster4 > 0 or n_outros > 0:
        _, p_value = fisher_exact(tabela)
        teste = 'Fisher'
        chi2 = np.nan
    else:
        p_value = 1.0
        teste = 'N/A'
        chi2 = np.nan
    
    # Salvar resultados
    resultados_causas.append({
        'Causa': causa,
        'N_Cluster4': n_cluster4,
        '%_Cluster4': prop_cluster4,
        'N_Outros': n_outros,
        '%_Outros': prop_outros,
        'Over-representation': over_rep,
        'Teste': teste,
        'p-valor': p_value,
        'Significativo': 'Sim' if p_value < 0.05 else 'Não'
    })

# Converter para DataFrame
df_causas = pd.DataFrame(resultados_causas)

# Ordenar por over-representation
df_causas_sorted = df_causas.sort_values('Over-representation', ascending=False)

# Salvar
df_causas_sorted.to_csv('cluster4_causas_analise.csv', index=False)
print("\n✓ Análise salva: cluster4_causas_analise.csv")

# =============================================================================
# 3. EXIBIR RESULTADOS
# =============================================================================

print("\n" + "-"*80)
print("CAUSAS MAIS PRESENTES NO CLUSTER 4 (Top 10)")
print("-"*80)

top10 = df_causas_sorted.head(10)

for idx, row in top10.iterrows():
    print(f"\n{row['Causa']}:")
    print(f"  Cluster 4: {row['N_Cluster4']} ({row['%_Cluster4']:.1f}%)")
    print(f"  Outros: {row['N_Outros']} ({row['%_Outros']:.1f}%)")
    print(f"  Over-representation: {row['Over-representation']:.2f}x")
    print(f"  p-valor: {row['p-valor']:.4f} ({row['Significativo']})")

# Causas estatisticamente significativas
print("\n" + "-"*80)
print("CAUSAS COM DIFERENÇA ESTATISTICAMENTE SIGNIFICATIVA (p<0.05)")
print("-"*80)

causas_sig = df_causas_sorted[df_causas_sorted['Significativo'] == 'Sim']

if len(causas_sig) > 0:
    for idx, row in causas_sig.iterrows():
        print(f"✓ {row['Causa']}: {row['Over-representation']:.2f}x (p={row['p-valor']:.4f})")
else:
    print("Nenhuma causa apresentou diferença estatisticamente significativa.")

# =============================================================================
# 4. VISUALIZAÇÕES
# =============================================================================

print("\n" + "="*80)
print("GERANDO VISUALIZAÇÕES")
print("="*80)

# Gráfico 1: Top 10 causas por over-representation
fig, ax = plt.subplots(figsize=(12, 8))

top10_plot = df_causas_sorted.head(10).copy()
top10_plot['Causa_short'] = top10_plot['Causa'].str[:40]  # Truncar para caber no gráfico

y_pos = np.arange(len(top10_plot))

# Barras
bars = ax.barh(y_pos, top10_plot['Over-representation'], color='darkred', alpha=0.7)

# Destacar causas significativas
for i, (idx, row) in enumerate(top10_plot.iterrows()):
    if row['Significativo'] == 'Sim':
        bars[i].set_color('firebrick')
        bars[i].set_alpha(1.0)

ax.set_yticks(y_pos)
ax.set_yticklabels(top10_plot['Causa_short'])
ax.set_xlabel('Over-representation (vezes mais frequente no Cluster 4)', fontsize=12)
ax.set_title('Top 10 Causas de Infraestrutura - Cluster 4 vs Outros Clusters', 
             fontsize=14, fontweight='bold')
ax.axvline(x=1, color='black', linestyle='--', linewidth=1, alpha=0.5, label='Baseline (1.0x)')

# Adicionar valores nas barras
for i, v in enumerate(top10_plot['Over-representation']):
    ax.text(v + 0.1, i, f'{v:.2f}x', va='center', fontsize=10)

ax.legend()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('cluster4_causas_top10.png', dpi=300, bbox_inches='tight')
print("✓ cluster4_causas_top10.png")
plt.close()

# Gráfico 2: Comparação de proporções (apenas causas presentes)
causas_presentes = df_causas_sorted[df_causas_sorted['N_Cluster4'] > 0].head(10)

fig, ax = plt.subplots(figsize=(12, 8))

x = np.arange(len(causas_presentes))
width = 0.35

bars1 = ax.bar(x - width/2, causas_presentes['%_Cluster4'], width, 
               label='Cluster 4', color='darkred', alpha=0.8)
bars2 = ax.bar(x + width/2, causas_presentes['%_Outros'], width, 
               label='Outros Clusters', color='steelblue', alpha=0.8)

ax.set_ylabel('Proporção de Acidentes (%)', fontsize=12)
ax.set_title('Comparação de Causas: Cluster 4 vs Outros Clusters', 
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(causas_presentes['Causa'].str[:30], rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('cluster4_causas_comparacao.png', dpi=300, bbox_inches='tight')
print("✓ cluster4_causas_comparacao.png")
plt.close()

# Gráfico 3: Heatmap de presença de causas
fig, ax = plt.subplots(figsize=(10, 12))

# Criar matriz de presença (top 15 causas)
top15_causas = df_causas_sorted.head(15)['Causa'].tolist()
matriz_presenca = []

for causa in top15_causas:
    cluster4_prop = (cluster4['causa_acidente'] == causa).mean() * 100
    outros_prop = (outros_clusters['causa_acidente'] == causa).mean() * 100
    matriz_presenca.append([cluster4_prop, outros_prop])

matriz = np.array(matriz_presenca)

sns.heatmap(matriz, annot=True, fmt='.1f', cmap='Reds', 
            xticklabels=['Cluster 4', 'Outros'],
            yticklabels=[c[:40] for c in top15_causas],
            cbar_kws={'label': 'Proporção (%)'},
            ax=ax)

ax.set_title('Presença de Causas: Cluster 4 vs Outros Clusters', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('cluster4_causas_heatmap.png', dpi=300, bbox_inches='tight')
print("✓ cluster4_causas_heatmap.png")
plt.close()

# =============================================================================
# 5. ANÁLISE DE COMBINAÇÕES (Top 3 causas)
# =============================================================================

print("\n" + "="*80)
print("ANÁLISE DE COMBINAÇÕES DE CAUSAS")
print("="*80)

# Pegar top 3 causas mais over-represented
top3_causas = df_causas_sorted.head(3)['Causa'].tolist()

print(f"\nTop 3 causas analisadas:")
for i, causa in enumerate(top3_causas, 1):
    print(f"{i}. {causa}")

# Criar flags para presença de cada causa
for causa in top3_causas:
    col_name = f"tem_{causa.lower().replace(' ', '_')[:20]}"
    cluster4[col_name] = (cluster4['causa_acidente'] == causa).astype(int)

# Analisar combinações
print(f"\nCombinações de causas no Cluster 4:")

for causa1 in top3_causas:
    col1 = f"tem_{causa1.lower().replace(' ', '_')[:20]}"
    for causa2 in top3_causas:
        if causa1 >= causa2:  # Evitar duplicatas
            continue
        col2 = f"tem_{causa2.lower().replace(' ', '_')[:20]}"
        
        # Contar co-ocorrências
        ambas = (cluster4[col1] == 1) & (cluster4[col2] == 1)
        n_ambas = ambas.sum()
        prop_ambas = (n_ambas / len(cluster4)) * 100
        
        if n_ambas > 0:
            print(f"\n{causa1} + {causa2}:")
            print(f"  {n_ambas} acidentes ({prop_ambas:.1f}%)")
            
            # Gravidade média dessa combinação
            grav_media = cluster4[ambas]['indice_gravidade'].mean()
            mortos_media = cluster4[ambas]['n_mortos'].mean()
            print(f"  Gravidade média: {grav_media:.2f}")
            print(f"  Mortos média: {mortos_media:.2f}")

# =============================================================================
# 6. RELATÓRIO EXECUTIVO
# =============================================================================

with open('CLUSTER4_CAUSAS_RELATORIO.txt', 'w', encoding='utf-8') as f:
    f.write("="*80 + "\n")
    f.write("RELATÓRIO: CAUSAS DE INFRAESTRUTURA - CLUSTER 4 (CRÍTICO)\n")
    f.write("="*80 + "\n\n")
    
    f.write("1. RESUMO\n")
    f.write("-"*80 + "\n")
    f.write(f"Cluster 4: {len(cluster4)} acidentes ({len(cluster4)/len(df)*100:.1f}% do total)\n")
    f.write(f"Mortalidade: {cluster4['n_mortos'].mean():.2f} mortos/acidente ")
    f.write(f"({cluster4['n_mortos'].mean() / df['n_mortos'].mean():.1f}x acima da média)\n\n")
    
    f.write("2. PRINCIPAIS CAUSAS DE INFRAESTRUTURA\n")
    f.write("-"*80 + "\n\n")
    
    for idx, row in top10.iterrows():
        f.write(f"• {row['Causa']}\n")
        f.write(f"  - Cluster 4: {row['N_Cluster4']} acidentes ({row['%_Cluster4']:.1f}%)\n")
        f.write(f"  - Over-representation: {row['Over-representation']:.2f}x\n")
        f.write(f"  - Significativo: {row['Significativo']} (p={row['p-valor']:.4f})\n\n")
    
    f.write("3. RECOMENDAÇÕES DE INTERVENÇÃO\n")
    f.write("-"*80 + "\n\n")
    
    f.write("Priorizar ações nas seguintes causas identificadas:\n\n")
    
    for idx, row in top10.head(5).iterrows():
        causa = row['Causa'].lower()
        f.write(f"→ {row['Causa']}:\n")
        
        # Sugestões específicas
        if 'iluminação' in causa:
            f.write("   - Instalar/melhorar iluminação pública\n")
            f.write("   - Manutenção preventiva de postes\n")
        elif 'pista' in causa and 'escorregadia' in causa:
            f.write("   - Melhorar drenagem\n")
            f.write("   - Reasfaltamento com material antiderrapante\n")
        elif 'água' in causa or 'drenagem' in causa:
            f.write("   - Modernizar sistema de drenagem\n")
            f.write("   - Limpeza regular de bueiros\n")
        elif 'sinalização' in causa:
            f.write("   - Revisar sinalização horizontal e vertical\n")
            f.write("   - Instalar placas reflexivas\n")
        elif 'acostamento' in causa:
            f.write("   - Construir/reformar acostamento\n")
            f.write("   - Demarcar com pintura refletiva\n")
        else:
            f.write("   - Avaliação técnica específica\n")
        
        f.write(f"   Impacto esperado: reduzir {row['N_Cluster4']} acidentes críticos\n\n")

print("\n✓ CLUSTER4_CAUSAS_RELATORIO.txt")

print("\n" + "="*80)
print("ANÁLISE CONCLUÍDA")
print("="*80)
print("\nArquivos gerados:")
print("  - cluster4_causas_analise.csv (dados brutos)")
print("  - cluster4_causas_top10.png (over-representation)")
print("  - cluster4_causas_comparacao.png (proporções)")
print("  - cluster4_causas_heatmap.png (mapa de calor)")
print("  - CLUSTER4_CAUSAS_RELATORIO.txt (relatório executivo)")


ANÁLISE DE CAUSAS DE INFRAESTRUTURA - CLUSTER 4 (CRÍTICO)

Cluster 4: 20 acidentes (8.5%)
Outros clusters: 216 acidentes (91.5%)

GRAVIDADE - Cluster 4:
  Total de mortos: 15
  Média de mortos/acidente: 0.73
  Proporção de acidentes fatais: 70.0%
  Índice médio de gravidade: 4.15

GRAVIDADE - Outros clusters:
  Média de mortos/acidente: 0.05
  Proporção de acidentes fatais: 4.2%
  Razão: 15.4x mais mortes

ANÁLISE DE CAUSAS DE INFRAESTRUTURA

✓ Análise salva: cluster4_causas_analise.csv

--------------------------------------------------------------------------------
CAUSAS MAIS PRESENTES NO CLUSTER 4 (Top 10)
--------------------------------------------------------------------------------

Pista esburacada:
  Cluster 4: 0 (0.0%)
  Outros: 0 (0.0%)
  Over-representation: infx
  p-valor: 1.0000 (Não)

Falta de acostamento:
  Cluster 4: 1 (5.0%)
  Outros: 0 (0.0%)
  Over-representation: infx
  p-valor: 0.0847 (Não)

Sistema de drenagem ineficiente:
  Cluster 4: 0 (0.0%)
  Outros: 0 (0.0%

In [7]:
# =============================================================================
# ANÁLISE DE CAUSAS DE INFRAESTRUTURA - TODOS OS CLUSTERS
# Objetivo: Identificar perfil de causas de infraestrutura em cada cluster
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, kruskal
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300

print("="*80)
print("ANÁLISE DE CAUSAS DE INFRAESTRUTURA POR CLUSTER")
print("="*80)

# =============================================================================
# 1. CARREGAR DADOS
# =============================================================================

df = pd.read_csv('2_acidentes_com_clusters.csv')

# Causas de infraestrutura
causas_infraestrutura = [
    'Iluminação deficiente', 'Demais falhas na via', 'Pista esburacada',
    'Falta de acostamento', 'Acesso irregular', 
    'Acumulo de água sobre o pavimento',
    'Acumulo de areia ou detritos sobre o pavimento',
    'Acumulo de óleo sobre o pavimento',
    'Falta de elemento de contenção que evite a saída do leito carroçável',
    'Restrição de visibilidade em curvas verticais',
    'Restrição de visibilidade em curvas horizontais',
    'Sistema de drenagem ineficiente', 'Declive acentuado',
    'Curva acentuada', 'Sinalização mal posicionada',
    'Desvio temporário', 'Ausência de sinalização',
    'Afundamento ou ondulação no pavimento',
    'Acostamento em desnível',
    'Deficiência do Sistema de Iluminação/Sinalização',
    'Pista Escorregadia'
]

n_clusters = df['cluster'].nunique()
print(f"\nTotal de clusters: {n_clusters}")
print(f"Total de acidentes: {len(df)}")

# Estatísticas por cluster
print("\n" + "-"*80)
print("TAMANHO E GRAVIDADE POR CLUSTER")
print("-"*80)

for cluster in sorted(df['cluster'].unique()):
    cluster_df = df[df['cluster'] == cluster]
    print(f"\nCluster {cluster}:")
    print(f"  N = {len(cluster_df)} ({len(cluster_df)/len(df)*100:.1f}%)")
    print(f"  Mortos/acidente: {cluster_df['n_mortos'].mean():.2f}")
    print(f"  Índice gravidade: {cluster_df['indice_gravidade'].mean():.2f}")
    print(f"  Acidentes fatais: {(cluster_df['n_mortos'] > 0).sum()} ({(cluster_df['n_mortos'] > 0).mean()*100:.1f}%)")

# =============================================================================
# 2. ANÁLISE DE CAUSAS POR CLUSTER
# =============================================================================

print("\n" + "="*80)
print("DISTRIBUIÇÃO DE CAUSAS POR CLUSTER")
print("="*80)

# Criar matriz: Clusters x Causas
matriz_causas = []
clusters_ids = sorted(df['cluster'].unique())

for cluster in clusters_ids:
    cluster_df = df[df['cluster'] == cluster]
    linha_causas = []
    
    for causa in causas_infraestrutura:
        n_causa = (cluster_df['causa_acidente'] == causa).sum()
        prop_causa = (n_causa / len(cluster_df)) * 100
        linha_causas.append(prop_causa)
    
    matriz_causas.append(linha_causas)

# DataFrame
df_matriz = pd.DataFrame(
    matriz_causas,
    index=[f'Cluster {c}' for c in clusters_ids],
    columns=[c[:30] for c in causas_infraestrutura]  # Truncar para visualização
)

# Salvar
df_matriz.to_csv('causas_por_cluster_matriz.csv')
print("\n✓ Matriz salva: causas_por_cluster_matriz.csv")

# =============================================================================
# 3. IDENTIFICAR CAUSAS DOMINANTES POR CLUSTER
# =============================================================================

print("\n" + "-"*80)
print("TOP 5 CAUSAS DOMINANTES POR CLUSTER")
print("-"*80)

causas_dominantes = []

for cluster in clusters_ids:
    cluster_df = df[df['cluster'] == cluster]
    
    print(f"\nCluster {cluster} (N={len(cluster_df)}):")
    
    # Contar causas
    causas_count = {}
    for causa in causas_infraestrutura:
        n = (cluster_df['causa_acidente'] == causa).sum()
        if n > 0:
            prop = (n / len(cluster_df)) * 100
            causas_count[causa] = {'N': n, 'Prop': prop}
    
    # Ordenar
    causas_sorted = sorted(causas_count.items(), key=lambda x: x[1]['Prop'], reverse=True)
    
    # Top 5
    for i, (causa, stats) in enumerate(causas_sorted[:5], 1):
        print(f"  {i}. {causa}: {stats['N']} ({stats['Prop']:.1f}%)")
        
        causas_dominantes.append({
            'Cluster': cluster,
            'Rank': i,
            'Causa': causa,
            'N': stats['N'],
            'Proporção_%': stats['Prop']
        })

# Salvar
df_dominantes = pd.DataFrame(causas_dominantes)
df_dominantes.to_csv('causas_dominantes_por_cluster.csv', index=False)
print("\n✓ Causas dominantes salvas: causas_dominantes_por_cluster.csv")

# =============================================================================
# 4. TESTES ESTATÍSTICOS (QUI-QUADRADO)
# =============================================================================

print("\n" + "="*80)
print("TESTES ESTATÍSTICOS - ASSOCIAÇÃO CAUSA x CLUSTER")
print("="*80)

resultados_testes = []

for causa in causas_infraestrutura:
    # Criar tabela de contingência: Clusters x (Tem causa / Não tem causa)
    tabela = []
    
    for cluster in clusters_ids:
        cluster_df = df[df['cluster'] == cluster]
        tem_causa = (cluster_df['causa_acidente'] == causa).sum()
        nao_tem_causa = len(cluster_df) - tem_causa
        tabela.append([tem_causa, nao_tem_causa])
    
    tabela = np.array(tabela)
    
    # Teste qui-quadrado (se houver casos suficientes)
    total_com_causa = tabela[:, 0].sum()
    
    if total_com_causa >= 5:
        try:
            chi2, p_value, dof, expected = chi2_contingency(tabela)
            
            # Cramér's V (medida de associação)
            n = tabela.sum()
            cramers_v = np.sqrt(chi2 / (n * (min(tabela.shape) - 1)))
            
            resultados_testes.append({
                'Causa': causa,
                'N_Total': total_com_causa,
                'Chi²': chi2,
                'p-valor': p_value,
                'Cramers_V': cramers_v,
                'Significativo': 'Sim' if p_value < 0.05 else 'Não'
            })
        except:
            pass

# DataFrame de resultados
df_testes = pd.DataFrame(resultados_testes)
df_testes = df_testes.sort_values('Cramers_V', ascending=False)

# Salvar
df_testes.to_csv('causas_testes_estatisticos.csv', index=False)
print("\n✓ Testes salvos: causas_testes_estatisticos.csv")

# Exibir causas significativas
print("\n" + "-"*80)
print("CAUSAS COM ASSOCIAÇÃO SIGNIFICATIVA AO CLUSTER (p<0.05)")
print("-"*80)

causas_sig = df_testes[df_testes['Significativo'] == 'Sim']

if len(causas_sig) > 0:
    for idx, row in causas_sig.iterrows():
        print(f"\n✓ {row['Causa']}:")
        print(f"  Chi² = {row['Chi²']:.2f}, p = {row['p-valor']:.4f}")
        print(f"  Cramér's V = {row['Cramers_V']:.3f} (força da associação)")
else:
    print("Nenhuma causa apresentou associação estatisticamente significativa.")

# =============================================================================
# 5. VISUALIZAÇÕES
# =============================================================================

print("\n" + "="*80)
print("GERANDO VISUALIZAÇÕES")
print("="*80)

# 1. Heatmap: Causas x Clusters
fig, ax = plt.subplots(figsize=(16, 12))

# Selecionar apenas causas com pelo menos 1% em algum cluster
causas_relevantes = df_matriz.columns[df_matriz.max() >= 1.0]
df_matriz_filtrado = df_matriz[causas_relevantes]

sns.heatmap(df_matriz_filtrado.T, annot=True, fmt='.1f', cmap='YlOrRd', 
            cbar_kws={'label': 'Proporção de Acidentes (%)'},
            linewidths=0.5, ax=ax)

ax.set_title('Distribuição de Causas de Infraestrutura por Cluster', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Cluster', fontsize=12)
ax.set_ylabel('Causa de Infraestrutura', fontsize=12)

plt.tight_layout()
plt.savefig('causas_heatmap_todos_clusters.png', dpi=300, bbox_inches='tight')
print("✓ causas_heatmap_todos_clusters.png")
plt.close()

# 2. Barplot empilhado: Top 5 causas por cluster
fig, axes = plt.subplots(2, 5, figsize=(20, 10))
axes = axes.ravel()

for i, cluster in enumerate(clusters_ids):
    cluster_df = df[df['cluster'] == cluster]
    
    # Top 5 causas desse cluster
    causas_count = {}
    for causa in causas_infraestrutura:
        n = (cluster_df['causa_acidente'] == causa).sum()
        if n > 0:
            causas_count[causa] = n
    
    # Ordenar
    causas_sorted = sorted(causas_count.items(), key=lambda x: x[1], reverse=True)[:5]
    
    if len(causas_sorted) > 0:
        causas_labels = [c[0][:25] for c in causas_sorted]
        causas_values = [c[1] for c in causas_sorted]
        
        axes[i].barh(causas_labels, causas_values, color=f'C{i}')
        axes[i].set_title(f'Cluster {cluster} (N={len(cluster_df)})', fontweight='bold')
        axes[i].set_xlabel('Número de Acidentes')
        axes[i].invert_yaxis()
        axes[i].grid(axis='x', alpha=0.3)

plt.suptitle('Top 5 Causas de Infraestrutura por Cluster', 
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('causas_top5_por_cluster.png', dpi=300, bbox_inches='tight')
print("✓ causas_top5_por_cluster.png")
plt.close()

# 3. Gráfico de barras agrupadas: Causas mais comuns
# Selecionar top 10 causas gerais
top10_causas_gerais = df['causa_acidente'].value_counts().head(10).index.tolist()
top10_causas_gerais = [c for c in top10_causas_gerais if c in causas_infraestrutura][:8]

if len(top10_causas_gerais) > 0:
    fig, ax = plt.subplots(figsize=(14, 8))
    
    x = np.arange(len(top10_causas_gerais))
    width = 0.08  # Largura de cada barra
    
    for i, cluster in enumerate(clusters_ids):
        cluster_df = df[df['cluster'] == cluster]
        valores = []
        
        for causa in top10_causas_gerais:
            prop = (cluster_df['causa_acidente'] == causa).mean() * 100
            valores.append(prop)
        
        offset = (i - n_clusters/2) * width
        ax.bar(x + offset, valores, width, label=f'Cluster {cluster}')
    
    ax.set_ylabel('Proporção de Acidentes (%)', fontsize=12)
    ax.set_title('Comparação de Causas Principais Entre Clusters', 
                 fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([c[:25] for c in top10_causas_gerais], rotation=45, ha='right')
    ax.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('causas_comparacao_clusters.png', dpi=300, bbox_inches='tight')
    print("✓ causas_comparacao_clusters.png")
    plt.close()

# 4. Cramér's V - Força de associação
if len(df_testes) > 0:
    fig, ax = plt.subplots(figsize=(12, 8))
    
    top_associacoes = df_testes.sort_values('Cramers_V', ascending=True).tail(15)
    
    cores = ['darkred' if x == 'Sim' else 'steelblue' for x in top_associacoes['Significativo']]
    
    ax.barh(range(len(top_associacoes)), top_associacoes['Cramers_V'], color=cores)
    ax.set_yticks(range(len(top_associacoes)))
    ax.set_yticklabels(top_associacoes['Causa'].str[:40])
    ax.set_xlabel("Cramér's V (Força da Associação)", fontsize=12)
    ax.set_title('Associação Entre Causas e Clusters (Top 15)', 
                 fontsize=14, fontweight='bold')
    ax.axvline(x=0.1, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='Fraca')
    ax.axvline(x=0.3, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='Moderada')
    ax.grid(axis='x', alpha=0.3)
    
    # Legenda
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='darkred', label='Significativo (p<0.05)'),
        Patch(facecolor='steelblue', label='Não significativo')
    ]
    ax.legend(handles=legend_elements, loc='lower right')
    
    plt.tight_layout()
    plt.savefig('causas_associacao_cramers_v.png', dpi=300, bbox_inches='tight')
    print("✓ causas_associacao_cramers_v.png")
    plt.close()

# =============================================================================
# 6. PERFIL NARRATIVO DE CADA CLUSTER
# =============================================================================

print("\n" + "="*80)
print("PERFIL NARRATIVO POR CLUSTER")
print("="*80)

perfis_narrativos = []

for cluster in clusters_ids:
    cluster_df = df[df['cluster'] == cluster]
    
    # Top 3 causas
    causas_count = {}
    for causa in causas_infraestrutura:
        n = (cluster_df['causa_acidente'] == causa).sum()
        if n > 0:
            prop = (n / len(cluster_df)) * 100
            causas_count[causa] = {'N': n, 'Prop': prop}
    
    causas_sorted = sorted(causas_count.items(), key=lambda x: x[1]['Prop'], reverse=True)[:3]
    
    # Estatísticas
    n = len(cluster_df)
    mortos_media = cluster_df['n_mortos'].mean()
    grav_media = cluster_df['indice_gravidade'].mean()
    fatais_perc = (cluster_df['n_mortos'] > 0).mean() * 100
    
    # Perfil categórico dominante
    tracado_dom = cluster_df['tracado_simplificado'].mode()[0] if 'tracado_simplificado' in cluster_df.columns else 'N/A'
    area_dom = cluster_df['area_tipo'].mode()[0] if 'area_tipo' in cluster_df.columns else 'N/A'
    periodo_dom = cluster_df['periodo_dia'].mode()[0] if 'periodo_dia' in cluster_df.columns else 'N/A'
    
    # Narrativa
    narrativa = f"""
Cluster {cluster} - "{get_cluster_name(cluster, mortos_media, causas_sorted)}"
{'='*70}
Tamanho: {n} acidentes ({n/len(df)*100:.1f}% do total)

GRAVIDADE:
- Mortos/acidente: {mortos_media:.2f}
- Índice de gravidade: {grav_media:.2f}
- Acidentes fatais: {fatais_perc:.1f}%

CAUSAS PRINCIPAIS:
"""
    
    for i, (causa, stats) in enumerate(causas_sorted, 1):
        narrativa += f"{i}. {causa}: {stats['N']} acidentes ({stats['Prop']:.1f}%)\n"
    
    narrativa += f"""
PERFIL VIÁRIO/AMBIENTAL DOMINANTE:
- Traçado: {tracado_dom}
- Área: {area_dom}
- Período: {periodo_dom}

CARACTERIZAÇÃO:
{get_caracterizacao(cluster, mortos_media, causas_sorted, tracado_dom, area_dom, periodo_dom)}
"""
    
    perfis_narrativos.append(narrativa)
    print(narrativa)

# Função auxiliar para nomear clusters
def get_cluster_name(cluster, mortos, causas):
    if mortos >= 0.5:
        return "CRÍTICO - Alta Letalidade"
    elif mortos >= 0.2:
        return "GRAVE - Mortalidade Moderada"
    elif mortos > 0:
        return "MODERADO - Baixa Mortalidade"
    else:
        return "LEVE - Sem Mortes"

# Função auxiliar para caracterização
def get_caracterizacao(cluster, mortos, causas, tracado, area, periodo):
    texto = ""
    
    if mortos >= 0.5:
        texto += "Cluster CRÍTICO com alta concentração de mortes. "
    
    if len(causas) > 0:
        causa_principal = causas[0][0].lower()
        
        if 'iluminação' in causa_principal or 'escorregadia' in causa_principal:
            texto += "Predominância de problemas noturnos e/ou climáticos. "
        elif 'sinalização' in causa_principal:
            texto += "Deficiências de sinalização são fator chave. "
        elif 'acostamento' in causa_principal or 'contenção' in causa_principal:
            texto += "Infraestrutura lateral deficiente. "
    
    if area == 'Rural':
        texto += "Concentrado em áreas rurais. "
    elif area == 'Urbana':
        texto += "Predominante em áreas urbanas. "
    
    if periodo == 'Noite':
        texto += "Risco elevado no período noturno. "
    
    return texto if texto else "Perfil genérico de acidentes de infraestrutura."

# Salvar perfis
with open('PERFIS_CLUSTERS_NARRATIVA.txt', 'w', encoding='utf-8') as f:
    f.write("="*80 + "\n")
    f.write("PERFIS NARRATIVOS DOS CLUSTERS\n")
    f.write("Acidentes de Infraestrutura - PRF 2024 - RIDE-DF\n")
    f.write("="*80 + "\n\n")
    
    for perfil in perfis_narrativos:
        f.write(perfil)
        f.write("\n" + "="*80 + "\n\n")

print("\n✓ PERFIS_CLUSTERS_NARRATIVA.txt")

# =============================================================================
# 7. RESUMO FINAL
# =============================================================================

print("\n" + "="*80)
print("ANÁLISE CONCLUÍDA")
print("="*80)
print("\nArquivos gerados:")
print("  Tabelas:")
print("    - causas_por_cluster_matriz.csv")
print("    - causas_dominantes_por_cluster.csv")
print("    - causas_testes_estatisticos.csv")
print("  Gráficos:")
print("    - causas_heatmap_todos_clusters.png")
print("    - causas_top5_por_cluster.png")
print("    - causas_comparacao_clusters.png")
print("    - causas_associacao_cramers_v.png")
print("  Relatórios:")
print("    - PERFIS_CLUSTERS_NARRATIVA.txt")


ANÁLISE DE CAUSAS DE INFRAESTRUTURA POR CLUSTER

Total de clusters: 10
Total de acidentes: 236

--------------------------------------------------------------------------------
TAMANHO E GRAVIDADE POR CLUSTER
--------------------------------------------------------------------------------

Cluster 0:
  N = 12 (5.1%)
  Mortos/acidente: 0.00
  Índice gravidade: 1.00
  Acidentes fatais: 0 (0.0%)

Cluster 1:
  N = 43 (18.2%)
  Mortos/acidente: 0.05
  Índice gravidade: 2.30
  Acidentes fatais: 2 (4.7%)

Cluster 2:
  N = 18 (7.6%)
  Mortos/acidente: 0.15
  Índice gravidade: 2.50
  Acidentes fatais: 2 (11.1%)

Cluster 3:
  N = 25 (10.6%)
  Mortos/acidente: 0.23
  Índice gravidade: 2.44
  Acidentes fatais: 5 (20.0%)

Cluster 4:
  N = 20 (8.5%)
  Mortos/acidente: 0.73
  Índice gravidade: 4.15
  Acidentes fatais: 14 (70.0%)

Cluster 5:
  N = 43 (18.2%)
  Mortos/acidente: 0.00
  Índice gravidade: 1.14
  Acidentes fatais: 0 (0.0%)

Cluster 6:
  N = 8 (3.4%)
  Mortos/acidente: 0.00
  Índice gravida

NameError: name 'get_cluster_name' is not defined

In [8]:
print("="*80)
print("DIAGNÓSTICO: QUALIDADE DO PESID")
print("="*80)

# 1. Valores únicos de pesid
print(f"\nTotal de registros: {len(df):,}")
print(f"pesid únicos: {df['pesid'].nunique():,}")

# 2. Proporção de pesid=0 ou nulos
pesid_zero = (df['pesid'] == 0).sum()
pesid_null = df['pesid'].isnull().sum()

print(f"\npesid = 0: {pesid_zero:,} ({pesid_zero/len(df)*100:.1f}%)")
print(f"pesid nulo: {pesid_null:,} ({pesid_null/len(df)*100:.1f}%)")

# 3. Verificar duplicatas suspeitas
# Agrupar por (id, pesid) e ver quantas linhas cada combinação tem
duplicatas = df.groupby(['id', 'pesid']).size()
duplicatas_suspeitas = duplicatas[duplicatas > 1]

print(f"\nCombinações (id, pesid) duplicadas: {len(duplicatas_suspeitas):,}")

if len(duplicatas_suspeitas) > 0:
    print(f"Exemplo de duplicatas:")
    print(duplicatas_suspeitas.head(10))

# 4. Analisar um acidente específico com duplicata
if len(duplicatas_suspeitas) > 0:
    id_exemplo, pesid_exemplo = duplicatas_suspeitas.index[0]
    
    print(f"\n" + "="*80)
    print(f"EXEMPLO: Acidente {id_exemplo}, Pessoa {pesid_exemplo}")
    print("="*80)
    
    exemplo = df[(df['id'] == id_exemplo) & (df['pesid'] == pesid_exemplo)]
    
    # Colunas relevantes
    cols_rel = ['id', 'pesid', 'estado_fisico', 'tipo_envolvido', 
                'causa_acidente', 'classificacao_acidente', 'tipo_acidente']
    cols_disponiveis = [c for c in cols_rel if c in exemplo.columns]
    
    print(exemplo[cols_disponiveis])
    
    # Verificar se são realmente a MESMA pessoa ou erro
    if 'estado_fisico' in exemplo.columns:
        estados_unicos = exemplo['estado_fisico'].nunique()
        print(f"\nEstados físicos diferentes: {estados_unicos}")
        
        if estados_unicos == 1:
            print("✓ Provável: MESMA pessoa em múltiplos eventos → drop_duplicates CORRETO")
        else:
            print("⚠️ ALERTA: Pessoa com estados físicos diferentes → Verificar!")

# 5. Verificar por acidente
print("\n" + "="*80)
print("ANÁLISE POR ACIDENTE")
print("="*80)

# Contar linhas vs pessoas únicas por acidente
linhas_por_acidente = df.groupby('id').size()
pessoas_por_acidente = df.groupby('id')['pesid'].nunique()

# Filtrar acidentes onde pesid_unico < linhas (indicando duplicatas)
acidentes_com_dup = (linhas_por_acidente != pessoas_por_acidente).sum()

print(f"Acidentes com pesid duplicados: {acidentes_com_dup:,} de {len(linhas_por_acidente):,}")
print(f"Proporção: {acidentes_com_dup/len(linhas_por_acidente)*100:.1f}%")

# Exemplos
acidentes_ids = (linhas_por_acidente != pessoas_por_acidente)
if acidentes_ids.any():
    exemplos_ids = acidentes_ids[acidentes_ids].index[:5]
    
    print(f"\nExemplos de acidentes com duplicatas:")
    for aid in exemplos_ids:
        print(f"  Acidente {aid}: {linhas_por_acidente[aid]} linhas, {pessoas_por_acidente[aid]} pessoas")

DIAGNÓSTICO: QUALIDADE DO PESID

Total de registros: 236


KeyError: 'pesid'